# Argomenti nel tempo — di cosa parlano oltre liturgia e fede

I notebook 01 e 04 partono da temi che proponiamo noi. Il 03 lasciava emergere i
temi ma su tutto il corpus, con etichette grezze. Qui la pipeline è più pulita, in
tre passi:

1. **marcare** ogni passaggio con una delle sei famiglie (campo `famiglia`
   nell'indice, scritto da `vdb.py arricchisci`);
2. **clusterizzare solo il non-core** — i passaggi *fuori* da liturgia e fede e
   devozione, cioè il ~20% dove si gioca la differenza; il fondo liturgico-devozionale
   lo sappiamo già continuo, non serve raggrupparlo;
3. **dare un nome leggibile** a ogni cluster leggendo i suoi passaggi
   rappresentativi (le parole grezze del TF-IDF non bastano).

Poi si contano i passaggi di ogni sotto-argomento **anno per anno**.

## Come gira

Richiede l'indice **arricchito** (campo `famiglia`: lancia `python vdb.py
arricchisci` nel repo del vector database). Il clustering è in **torch** (l'env
conda con MKL crasha su `KMeans`/numpy-matmul). Niente testi stampati: solo
conteggi.

In [ ]:
import collections
import numpy as np
import torch
from ponte import tabella

tab = tabella()
w = "escludibile=false AND famiglia NOT IN ('liturgia','fede e devozione')"
t = tab.search().where(w).select(["data", "vector"]).limit(10**9).to_arrow()
anno = np.array([d[:4] if d else "" for d in t.column("data").to_pylist()])
V = torch.from_numpy(np.stack(t.column("vector").to_pylist()).astype("float32"))
print(len(V), "passaggi non-core (fuori da liturgia e fede)")

In [ ]:
def kmeans(X, k, iters=30, seed=0):
    torch.manual_seed(seed)
    C = X[torch.randperm(len(X))[:k]].clone()
    for _ in range(iters):
        lab = (X @ C.T).argmax(1)
        for j in range(k):
            s = X[lab == j]
            if len(s):
                c = s.mean(0); C[j] = c / (c.norm() + 1e-9)
    return lab.numpy()

K = 15
lab = kmeans(V, K)

# Etichette assegnate LEGGENDO i passaggi rappresentativi di ogni cluster (il
# passo "LLM/umano"): le parole grezze del TF-IDF erano illeggibili. Valide per
# seed=0 e questo `where`; se cambi seed/k vanno riassegnate rileggendo i campioni.
ETICHETTE = {
    1: "Visite ad limina dei vescovi", 10: "Vescovi e formazione del clero",
    8: "Udienze ufficiali (cardinali, accademie)", 13: "Viaggi apostolici e Chiese locali",
    5: "Saluti ai Paesi e diplomazia", 14: "Viaggi pastorali e benedizioni",
    0: "Dottrina sociale e morale", 3: "Povertà, lavoro e sviluppo",
    9: "Pace, disarmo e relazioni internazionali", 2: "Fame, agricoltura e sviluppo (FAO)",
    11: "Memoria della Chiesa (predecessori, Concilio)", 12: "Università, cultura e sapere",
    6: "Polonia e Divina Misericordia", 4: "Anziani, famiglia e spiritualità",
    7: "Francesco a braccio (interviste, dialoghi)",
}
for c in sorted(range(K), key=lambda c: -(lab == c).sum()):
    print(f"{int((lab==c).sum()):6}  {ETICHETTE[c]}")

## La heatmap: sotto-argomento × anno

Ogni colonna è un anno, ogni riga un sotto-argomento, il colore la sua quota tra i
passaggi non-core di quell'anno. Righe piene = argomenti che restano; macchie =
argomenti a ondate. Argomenti ordinati per anno di picco; linee azzurre = cambi di
Papa.

In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

anni = sorted({y for y in anno if y})
tot = {y: int((anno == y).sum()) for y in anni}
anni = [y for y in anni if tot[y] >= 30]
labels = [ETICHETTE[c] for c in range(K)]
M = np.array([[100 * ((lab == c) & (anno == y)).sum() / tot[y] for y in anni] for c in range(K)])
ordine = np.argsort([int(anni[M[c].argmax()]) for c in range(K)])
M, labels = M[ordine], [labels[i] for i in ordine]

fig, ax = plt.subplots(figsize=(13, 7))
sns.heatmap(M, ax=ax, cmap="rocket_r", yticklabels=labels, xticklabels=anni,
            cbar_kws={"label": "% dei passaggi non-core dell'anno"})
for i, y in enumerate(anni):
    if y in ("2005", "2013", "2025"):
        ax.axvline(i, color="cyan", lw=1.2)
ax.set_xlabel("anno  (linee azzurre: cambio di Papa — 2005, 2013, 2025)")
ax.set_title("Di cosa parlano i Papi oltre liturgia e fede — argomenti nel tempo\n"
             "(righe piene = continui · macchie = a ondate)", fontsize=12)
plt.setp(ax.get_xticklabels(), fontsize=7, rotation=90)
plt.setp(ax.get_yticklabels(), fontsize=9, rotation=0)
fig.tight_layout(); fig.savefig("argomenti-nel-tempo.png", dpi=150, bbox_inches="tight")
print("salvata argomenti-nel-tempo.png")

![argomenti non-core nel tempo](argomenti-nel-tempo.png)

## Come si legge

Due cose, oneste, prima della storia:
- anche il non-core è per **circa metà istituzionale** — visite *ad limina*, udienze
  ufficiali, viaggi e saluti diplomatici: argomenti dettati dal ruolo, non scelti;
- gran parte del volume è di **Giovanni Paolo II**, che ha prodotto molto più testo
  degli altri — quindi le righe vanno lette come *quote*, non come numeri assoluti.

Detto questo, gli argomenti **di contenuto** si muovono, e quasi sempre al cambio di
Papa: l'**America Latina / ad limina** e la **Polonia** sono dense negli anni di
Giovanni Paolo II e si schiariscono dopo; **Povertà-lavoro**, **Fame/FAO** e il
registro **"Francesco a braccio"** si accendono con Francesco e restano con Leone;
**Pace e disarmo** torna a tratti, legato ai conflitti del momento. Il fondo
(liturgia e fede, qui escluso apposta) resta invece continuo: è la continuità di
cui parlano gli altri notebook.

---
*Argomenti **estratti dal dato** (raggruppamento per significato sul solo non-core),
con nomi assegnati leggendo i passaggi rappresentativi. Solo conteggi e quote per
anno, nessun testo riprodotto (© Libreria Editrice Vaticana).*